In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, ConcatDataset
import tifffile as tiff
import os
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.manifold import TSNE
from PIL import Image
from tqdm import tqdm
from torchvision import transforms

from skimage import io
import sys
# from umap import UMAP
import joblib
from datetime import datetime

# Load Data
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [2]:
# Get the directory of the script
script_dir = os.getcwd()

# Get the parent directory of the script
parent_dir = os.path.dirname(script_dir)

# Add the parent directory to sys.path
sys.path.append(parent_dir)

from core.autoencoders import AE, train_ae
from core.dataset import TIFFDataset
from utils.feature_analysis import UMAP_train, dataloader_model_latents,kmeans_cluster,DBSCAN_cluster
from utils.plotting_utils import umap_2Dplot, cluster_2Dplot



In [3]:
data_str = 'vin_pax_zyx_act_front'
pro_str = 'vin_front_correctloader_lossnorm'
main_ch = 1
ctrl_y_str = 'ctrl_y'
input_ps = 32
latent_dim_array = [4,5,6,7,8,9,10,12,14,16,18,20,22,24,26,28,30,32]
BN_flag = True
dropout_flag = True
epochs = int(10000)
lr = 1e-4
eps = 3
min_samples = 5
kmeans_num_clusters = 6
loss_norm_flag = 1

dir_list = [
'/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/ctrl_ch0_major/patches_gridonly_pslocation00/tiff_patches32_65p_20250909_1518',
'/mnt/d/lding/FA/analysis_results/FA_ML_Annabel_20250217/031125/y_ch0_major/y_ch0_patches_gridonly_pslocation00/tiff_patches32_65p_20250909_1530',
]



In [4]:

transform = transforms.Compose([transforms.ToTensor()])

datasets = [
    TIFFDataset(root_dir=dir_path, label=label, transform=transform)
    for label, dir_path in enumerate(dir_list)
]

combined_dataset = ConcatDataset(datasets)

# Split dataset into training and validation sets
train_size = int(0.8 * len(combined_dataset))
val_size = len(combined_dataset) - train_size
train_dataset, val_dataset = random_split(combined_dataset, [train_size, val_size])
whole_data_loader = DataLoader(combined_dataset, batch_size=128, shuffle=True)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)



/mnt/d/lding/CLS_GitHub/fa_patch_AE_clustering/core/dataset.py:33: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  image = torch.tensor(image, dtype=torch.float32)  # shape: (1, H, W)


In [5]:
total_sum_sq = 0.0
total_sum = 0.0
n_total = 0

for images, _ in whole_data_loader:
    images = images.to(torch.float32)
    total_sum += (images).sum().item()
    total_sum_sq += (images ** 2).sum().item()
    n_total += images.numel()

mean_square = total_sum_sq / n_total
mean = total_sum / n_total
print("Mean square of all pixels:", mean_square)

joblib.dump(mean_square,'../results/vin_front_mean_square.pkl')

Mean square of all pixels: 0.010358415582496418


['../results/vin_front_mean_square.pkl']

In [ ]:
train_loss_across_latentdim = []
val_loss_across_latentdim = []

for  latent_dim in latent_dim_array:
        
    now = datetime.now()

    time_str = now.strftime("%Y%m%d_%H%M")

    result_dir = os.path.join('../results/', pro_str+'_ch' + str(main_ch) +'_'+ 'ps'+str(input_ps)+'_latdim_'+ str(latent_dim) +'_'+ time_str)
    os.makedirs(result_dir, exist_ok=True)

    # Train AE
    ae = AE(latent_dim=latent_dim, input_ps=input_ps, BN_flag=BN_flag, dropout_flag=dropout_flag).to(device)

    ae, train_losses, val_losses = train_ae(ae, train_loader, val_loader, device, epochs=epochs, lr=lr, loss_norm_flag=loss_norm_flag, result_dir = result_dir)

    train_loss_across_latentdim.append(train_losses[-1])
    val_loss_across_latentdim.append(val_losses[-1])
    
    latents, images, group_id = dataloader_model_latents(ae, whole_data_loader, device)

    if isinstance(group_id, torch.Tensor):
        group_id = group_id.cpu().numpy()
    elif isinstance(group_id, list):
        group_id = torch.cat(group_id).cpu().numpy()

    latents_2d = UMAP_train(latents, result_dir)

    DBSCAN, DBSCAN_labels =  DBSCAN_cluster(latents, eps=eps, min_samples=min_samples, result_dir=result_dir)

    kmeans, kmeans_labels = kmeans_cluster(latents, num_clusters=kmeans_num_clusters, result_dir=result_dir)

    fig = umap_2Dplot(latents_2d, 0,1,group_id)
    fig.savefig(os.path.join(result_dir, 'umap_2d_grouplabels.png'))

    fig = cluster_2Dplot(latents, 0,1,DBSCAN_labels)
    fig.savefig(os.path.join(result_dir, 'DBSCAN_latent01_labels.png'))
    
    fig = cluster_2Dplot(latents, 0,1,kmeans_labels)
    fig.savefig(os.path.join(result_dir, 'kmeans_latent01_labels.png'))

    fig = cluster_2Dplot(latents_2d, 0,1,DBSCAN_labels)
    fig.savefig(os.path.join(result_dir, 'DBSCAN_umap2d_labels.png'))
    
    fig = cluster_2Dplot(latents_2d, 0,1,kmeans_labels)
    fig.savefig(os.path.join(result_dir, 'kmeans_umap2d_labels.png'))


Epoch 200/10000, Train Loss: 0.3964, Val Loss: 0.4489
Epoch 400/10000, Train Loss: 0.3671, Val Loss: 0.4655
Epoch 600/10000, Train Loss: 0.3530, Val Loss: 0.4776
Epoch 800/10000, Train Loss: 0.3368, Val Loss: 0.4857
Epoch 1000/10000, Train Loss: 0.3289, Val Loss: 0.4929
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0001, max: 0.5419, mean: 0.0496, std: 0.0442
Epoch 1200/10000, Train Loss: 0.3244, Val Loss: 0.4996
Epoch 1400/10000, Train Loss: 0.3144, Val Loss: 0.5051
Epoch 1600/10000, Train Loss: 0.3099, Val Loss: 0.5108
Epoch 1800/10000, Train Loss: 0.3069, Val Loss: 0.5169
Epoch 2000/10000, Train Loss: 0.3026, Val Loss: 0.5173
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0000, max: 0.6103, mean: 0.0502, std: 0.0437
Epoch 2200/10000, Train Loss: 0.3009, Val Loss: 0.5201
Epoch 2400/10000, Train Loss: 0.2954, Val Loss: 0.5256
Epoch 2600/10000, Train Loss: 0.2912, Val Loss: 0.5279
Epoch 2

/home/ldin/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Epoch 200/10000, Train Loss: 0.3719, Val Loss: 0.4307
Epoch 400/10000, Train Loss: 0.3381, Val Loss: 0.4413
Epoch 600/10000, Train Loss: 0.3216, Val Loss: 0.4547
Epoch 800/10000, Train Loss: 0.3078, Val Loss: 0.4652
Epoch 1000/10000, Train Loss: 0.2970, Val Loss: 0.4735
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0001, max: 0.9003, mean: 0.0519, std: 0.0467
Epoch 1200/10000, Train Loss: 0.2897, Val Loss: 0.4822
Epoch 1400/10000, Train Loss: 0.2802, Val Loss: 0.4878
Epoch 1600/10000, Train Loss: 0.2739, Val Loss: 0.4890
Epoch 1800/10000, Train Loss: 0.2661, Val Loss: 0.4939
Epoch 2000/10000, Train Loss: 0.2644, Val Loss: 0.5007
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0000, max: 0.8946, mean: 0.0507, std: 0.0465
Epoch 2200/10000, Train Loss: 0.2575, Val Loss: 0.5037
Epoch 2400/10000, Train Loss: 0.2555, Val Loss: 0.5069
Epoch 2600/10000, Train Loss: 0.2530, Val Loss: 0.5080
Epoch 2

/mnt/d/lding/CLS_GitHub/fa_patch_AE_clustering/core/autoencoders.py:97: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  fig, axes = plt.subplots(2, n, figsize=(n * 1, 2))


Epoch 5200/10000, Train Loss: 0.2284, Val Loss: 0.5304
Epoch 5400/10000, Train Loss: 0.2260, Val Loss: 0.5402
Epoch 5600/10000, Train Loss: 0.2261, Val Loss: 0.5329
Epoch 5800/10000, Train Loss: 0.2245, Val Loss: 0.5368
Epoch 6000/10000, Train Loss: 0.2204, Val Loss: 0.5322
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0000, max: 0.7834, mean: 0.0470, std: 0.0461
Epoch 6200/10000, Train Loss: 0.2221, Val Loss: 0.5454
Epoch 6400/10000, Train Loss: 0.2194, Val Loss: 0.5329
Epoch 6600/10000, Train Loss: 0.2214, Val Loss: 0.5355
Epoch 6800/10000, Train Loss: 0.2226, Val Loss: 0.5423
Epoch 7000/10000, Train Loss: 0.2175, Val Loss: 0.5394
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0000, max: 0.7423, mean: 0.0493, std: 0.0475
Epoch 7200/10000, Train Loss: 0.2182, Val Loss: 0.5414
Epoch 7400/10000, Train Loss: 0.2173, Val Loss: 0.5423
Epoch 7600/10000, Train Loss: 0.2175, Val Loss: 0.5418
Epo

...

/home/ldin/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Epoch 200/10000, Train Loss: 0.2291, Val Loss: 0.2608
Epoch 400/10000, Train Loss: 0.1800, Val Loss: 0.2790
Epoch 600/10000, Train Loss: 0.1608, Val Loss: 0.2900
Epoch 800/10000, Train Loss: 0.1454, Val Loss: 0.3003
Epoch 1000/10000, Train Loss: 0.1414, Val Loss: 0.3087
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0000, max: 0.6707, mean: 0.0523, std: 0.0551
Epoch 1200/10000, Train Loss: 0.1324, Val Loss: 0.3092
Epoch 1400/10000, Train Loss: 0.1295, Val Loss: 0.3164
Epoch 1600/10000, Train Loss: 0.1303, Val Loss: 0.3184
Epoch 1800/10000, Train Loss: 0.1230, Val Loss: 0.3212
Epoch 2000/10000, Train Loss: 0.1217, Val Loss: 0.3224
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0000, max: 0.6671, mean: 0.0501, std: 0.0534
Epoch 2200/10000, Train Loss: 0.1212, Val Loss: 0.3277
Epoch 2400/10000, Train Loss: 0.1200, Val Loss: 0.3281
Epoch 2600/10000, Train Loss: 0.1178, Val Loss: 0.3298
Epoch 2

/home/ldin/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Epoch 200/10000, Train Loss: 0.2281, Val Loss: 0.2563
Epoch 400/10000, Train Loss: 0.1768, Val Loss: 0.2733
Epoch 600/10000, Train Loss: 0.1580, Val Loss: 0.2869
Epoch 800/10000, Train Loss: 0.1428, Val Loss: 0.2916
Epoch 1000/10000, Train Loss: 0.1346, Val Loss: 0.2981
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0002, max: 0.6528, mean: 0.0508, std: 0.0556
Epoch 1200/10000, Train Loss: 0.1334, Val Loss: 0.3037
Epoch 1400/10000, Train Loss: 0.1294, Val Loss: 0.3097
Epoch 1600/10000, Train Loss: 0.1253, Val Loss: 0.3146
Epoch 1800/10000, Train Loss: 0.1220, Val Loss: 0.3158
Epoch 2000/10000, Train Loss: 0.1219, Val Loss: 0.3164
Input stats — min: 0.0000, max: 0.9486, mean: 0.0530, std: 0.0788
Reconstruction stats — min: 0.0001, max: 0.5867, mean: 0.0501, std: 0.0531
Epoch 2200/10000, Train Loss: 0.1192, Val Loss: 0.3190
Epoch 2400/10000, Train Loss: 0.1182, Val Loss: 0.3223
Epoch 2600/10000, Train Loss: 0.1171, Val Loss: 0.3252
Epoch 2

/home/ldin/.local/lib/python3.10/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [ ]:
joblib.dump(train_loss_across_latentdim, os.path.join('../results', 'train_loss_across_latentdim'+pro_str+'_ch' + str(main_ch) +'_'+ 'ps'+str(input_ps)+'_latdim_'+ str(latent_dim) +'_'+ time_str+'.pkl'))
joblib.dump(val_loss_across_latentdim, os.path.join('../results', 'val_loss_across_latentdim'+pro_str+'_ch' + str(main_ch) +'_'+ 'ps'+str(input_ps)+'_latdim_'+ str(latent_dim) +'_'+ time_str+'.pkl'))
    

In [ ]:
# Plot training and validation loss
fig = plt.figure(figsize=(8, 6))
plt.plot(latent_dim_array, train_loss_across_latentdim, label='Train Loss')
plt.plot(latent_dim_array, val_loss_across_latentdim, label='Validation Loss')
plt.xlabel('Latent dim')
plt.ylabel('Loss')
plt.legend()
plt.title('Training vs Validation Loss')
fig.savefig(os.path.join('../results', 'latent_dim_train_val_losses'+pro_str+'_ch' + str(main_ch) +'_'+ 'ps'+str(input_ps)+'_latdim_'+ str(latent_dim) +'_'+ time_str+'.png'))